# Hartland Quay (Devon) — LDI meshes

The isosurfaces of the LDI fit for the 3-D figures, from the model
`05b_hartland_quay_ldi` saved (`results/restore.pt`). Nothing is fitted here.

`MEMBER` picks which ensemble member to mesh (0 is the 05b fit, `restore.pt`; any other is
`results/ensemble/member_NN.pt`). For member 0 the surfaces are the fitted traces 05b chose and stored in
`results/beds.npz`; any other member chooses its own best-spread traces by the same rule. Each is meshed at that trace's restored depth, so the
sheets are the contacts of those beds, youngest first. Surfaces are octree dual
contours of the field (`GeoEvent.mesh`), saved as plain arrays to
`results/meshes_ldi.npz` (member 0; other members write `meshes_ldi_member_NN.npz`, which the
figure notebooks do not read unless you point them at it). The LDI fits have no interior extrema (05b checks), so there are no shells to mesh; the
Surfe meshes, which do, are written by `05c`.

## Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import torch

import curlew
from curlew.io import loadModel

import sys
sys.path.insert(0, str(Path("..").resolve()))  # publication/: figstyle, diagnostics, exploded3d
import diagnostics
import hartland_inputs

curlew.device = "cpu"
curlew.dtype = torch.float32
curlew.default_dim = 3

MEMBER = 19                 # which ensemble member to mesh: 0 is the 05b fit, k is results/ensemble/member_kk.pt

RES_DIR = hartland_inputs.RESULTS_DIR
inp = np.load(RES_DIR / "inputs.npz")
model = loadModel(RES_DIR / "restore.pt" if MEMBER == 0 else RES_DIR / "ensemble" / f"member_{MEMBER:02d}.pt")
event = model.events[0]
print(f"member {MEMBER}")

member 19


## Isosurfaces

The levels are the traces in `results/beds.npz`, not an even spacing of every
trace. `MESH_DEPTH` is the octree depth (2^depth cells per axis). The box is the
evaluation grid with a `PAD` margin. For each interior extremum, the level
surface `SHELL_DELTA` of the level spacing inside it is meshed within
`SHELL_HALF` m of it, as in 05c.

In [2]:
MESH_DEPTH = 7
QEF_TOL = 0.5              # dual-contouring vertex: 1 = mass point (smooth sheets); 0.1 = keep sharp features
PAD = 10.0                 # m
SHELL_DELTA = 0.08         # shell level, as a fraction of the median gap between the beds
SHELL_HALF = 15.0          # m

N_BEDS = 12                # beds for a member other than 0, as in 05b


def _spread(depths, targets):
    """Trace indices paired, in order, to target depths, each trace used once (as in 05b)."""
    depths = np.asarray(depths, float)
    targets = np.sort(np.asarray(targets, float))
    order = np.argsort(depths, kind="mergesort")
    d = depths[order]
    m, n = len(targets), len(d)
    cost = np.empty((m, n))
    prev = np.full((m, n), -1, int)
    cost[0] = np.abs(d - targets[0])
    for j in range(1, m):
        best, arg = np.inf, -1
        for i in range(n):
            if i > 0 and cost[j - 1, i - 1] < best:
                best, arg = cost[j - 1, i - 1], i - 1
            cost[j, i] = np.inf if arg < 0 else abs(d[i] - targets[j]) + best
            prev[j, i] = arg
    chosen = np.empty(m, int)
    i = int(np.argmin(cost[-1]))
    for j in range(m - 1, -1, -1):
        chosen[j] = order[i]
        i = int(prev[j, i])
    return chosen


def _scalar(xyz):
    with torch.no_grad():
        return np.concatenate([
            np.asarray(model.predict(torch.as_tensor(c, dtype=curlew.dtype), litho=False).scalar).ravel()
            for c in np.array_split(np.asarray(xyz, float), max(1, len(xyz) // 50_000))])


if MEMBER == 0:
    bed_path = RES_DIR / "beds.npz"
    if not bed_path.is_file():
        raise FileNotFoundError(f"{bed_path} is missing; run the lithology cell in 05b")
    beds = np.load(bed_path)
    bed_names = [str(n) for n in beds["names"]]
    levels = np.asarray(beds["isovalues"], float)
else:
    # this member's own best-spread beds: traces nearest equal shares of its cloud depths
    trace_names = [str(n) for n in inp["horizon_names"]]
    traces = [inp[f"horizon_{i}"] for i in range(len(trace_names))]
    off = np.cumsum([0, *[len(t) for t in traces]])
    phi_t = _scalar(np.vstack(traces))
    depths = np.array([float(phi_t[off[i]:off[i + 1]].mean()) for i in range(len(traces))])
    targets = np.quantile(_scalar(inp["cloud_xyz"]), (np.arange(N_BEDS) + 0.5) / N_BEDS)
    pick = _spread(depths, targets)
    bed_names = [trace_names[k] for k in pick]
    levels = depths[pick]
order = np.argsort(-levels, kind="mergesort")   # youngest (highest restored depth) first
bed_names = [bed_names[i] for i in order]
levels = levels[order]
spacing = float(np.median(np.diff(np.sort(levels))))
print("beds, youngest first:", ", ".join(f"{n} {v:.1f}" for n, v in zip(bed_names, levels)),
      f"| median gap {spacing:.1f} m")

axes = (inp["x"], inp["y"], inp["z"])
box = np.array([[a[0] - PAD for a in axes], [a[-1] + PAD for a in axes]])


def mesher(level, bounds):
    v, f = event.mesh(iso=float(level), bounds=np.asarray(bounds), max_depth=MESH_DEPTH, qef_tol=QEF_TOL)
    return np.asarray(v), np.asarray(f)


extrema = []               # none: the fits have no interior extrema (checked in 05b), so no shells
mesh_path = RES_DIR / ("meshes_ldi.npz" if MEMBER == 0 else f"meshes_ldi_member_{MEMBER:02d}.npz")
diagnostics.export_meshes(mesh_path, mesher, levels, box, extrema=extrema,
                          shell_delta=SHELL_DELTA * spacing, shell_half=SHELL_HALF,
                          shell_tries=4 if extrema else 0)
with np.load(mesh_path) as saved:
    arrays = {k: saved[k] for k in saved.files}
arrays["names"] = np.asarray(bed_names)
np.savez_compressed(mesh_path, **arrays)
m = np.load(mesh_path)
print({h: len(m[f"sheet{h}_f"]) for h in range(len(levels))}, "faces per surface;", len(extrema), "extrema")

beds, youngest first: H8 80.3, H27 60.6, H31 48.1, H45 38.3, H21 30.5, H63 23.7, H17 18.0, H55 12.5, H117 6.1, H37 2.1, H86 -1.3, H95 -5.9 | median gap 6.4 m
{0: 109318, 1: 117824, 2: 116406, 3: 114016, 4: 110432, 5: 107222, 6: 103890, 7: 99542, 8: 94724, 9: 91388, 10: 87892, 11: 79532} faces per surface; 0 extrema
